In [ ]:
%matplotlib inline
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
import seaborn as sns
from programmable_cubes_UDP import ProgrammableCubes
from programmable_cubes_UDP import programmable_cubes_UDP
import numpy as np
from pygmo import problem
import random
from numba import njit

## Load

In [ ]:


udp = programmable_cubes_UDP("ISS_INV")
prob = problem(udp)
print(f"Reference fitness: {udp.fitness(np.array([-1]))}")
ti = udp.initial_cube_types
ci = udp.final_cube_positions
ct = udp.target_cube_positions
tt = udp.target_cube_types
types = np.arange(np.max(ti)+1)

print("TARGET")
udp.plot("none",cube_type_to_plot=types,custom_config=udp.target_cube_positions,custom_cube_types=udp.target_cube_types)
print("INITIAL")
udp.plot("none",cube_type_to_plot=types,custom_config=udp.final_cube_positions,custom_cube_types=udp.initial_cube_types)

def plot(c,t):
    udp.plot("none",cube_type_to_plot=types,custom_config=c,custom_cube_types=t)
        
def debug_plot(c,t):
    udp.debug_plot("none",cube_type_to_plot=types,custom_config=c,custom_cube_types=t)
        


cubes = ProgrammableCubes(ci)



## Experiments to understand intesection and wrong cubes

In [ ]:

def contains_coord(arr:np.ndarray, coord:np.ndarray):
    for id,c in enumerate(arr):
        if np.sum(c==coord) == len(coord):
            return id
    return -1



# Get the wrong ones
# print(ci.shape)
# print(ct.shape)
all = np.concat([ci,ct])
# print(all.shape)
# This finds coordinates of cubes which are not in their place or empty target place(IGNORES types)
arr, uniq_cnt = np.unique(all, axis=0, return_counts=True)
wrong = arr[uniq_cnt==1]
#print(wrong)
# Find empty target place ids:
wrong_ci_ids = []
for coord in wrong:
    id = contains_coord(ci,coord) 
    if id != -1:
        wrong_ci_ids.append(id)
wrong_ci_ids = np.array(wrong_ci_ids)
print(wrong_ci_ids)
a = np.where(ci==wrong[0])
print(wrong[0])
print(ci)
print(a[0])
print(a[1])

## Useful functions

In [ ]:

def is_accessible(c,positions)->bool:
    """ Checks if there is at least one cube in some direction """
    # Target one has to be empty
    idx = np.where((positions == [c[0],c[1],c[2]]).all(axis=1))[0]
    if idx.size > 0:
        return False
    # One of the adjacent ones has to be filled
    DIRS = [[1,0,0],[-1,0,0],[0,1,0],[0,-1,0],[0,0,1],[0,0,-1]]
    for d in DIRS:
        idx = np.where((positions == [c[0]+d[0],c[1]+d[1],c[2]+d[2]]).all(axis=1))[0]
        if idx.size > 0:
            return True
    return False

loc_rots = [0,1,2,3,4,5]
def force_random_move(id,cubes:programmable_cubes_UDP,rand : int = 0):
    for i in loc_rots:
        res = cubes.apply_single_update_step(id,(i+rand)%6)
        if res == 1:
            return i
    return -1

ROTS = [0,1,2,3,4,5]
ROT_X = [2,3,4,5]
ROT_Y = [0,1,2,3]
ROT_Z = [0,1,4,5]
ROTS_XYZ = [ROT_X,ROT_Y,ROT_Z]
@njit(cache=True)
def inv_rot(rot:int)->int:
    match rot:
        case 0:
            return 1
        case 1:
            return 0
        case 2:
            return 3
        case 3:
            return 2
        case 4:
            return 5
        case 5:
            return 4

@njit(cache=True)
def dist_metric(a,b):
    return np.sum(np.abs(a-b))

def get_valid_rots(id,cubes:programmable_cubes_UDP):
    valid_rots = []
    for i in loc_rots:
        res = cubes.apply_single_update_step(id,i)
        if res == 1:
            cubes.apply_single_update_step(id,inv_rot(i))
            valid_rots.append(i)
    return valid_rots





#is_accessible([5,5,5],cubes)

## Intersection and difference implementation for algorithm

In [ ]:
def have_wrong_type(ci,ct,ti,tt):
    wrong_type_ids = []
    for i in np.arange(len(ci)):
        id = contains_coord(ct,ci[i])
        if id != -1 and ti[i] != tt[id]:
            wrong_type_ids.append(i)
    return wrong_type_ids

def get_wrong_cube_ids(arr1:np.ndarray,arr2:np.ndarray):
    """
    does not include color
    """
    arr, cnt = np.unique(np.concat([arr1,arr2]), axis=0, return_counts=True)
    wrong = arr[cnt==1]
    arr1_ids = []
    for coord in wrong:
        id = contains_coord(arr1,coord) 
        if id != -1:
            arr1_ids.append(id)
    arr1_ids = np.array(arr1_ids)

    arr2_ids = []
    for coord in wrong:
        id = contains_coord(arr2,coord) 
        if id != -1:
            arr2_ids.append(id)
    arr2_ids = np.array(arr2_ids)

    return arr1_ids,arr2_ids

print(get_wrong_cube_ids(ci,ct))

## Finding path from coord to coord

### Random walk

In [ ]:
wi,wt = get_wrong_cube_ids(ci,ct)
# Pick 1 coord of wrong cube and 1 coord of empty target
c1 = ci[wi[0]]
c2 = ct[wt[0]]

# Find the path
alg = []
plot(ct,tt)
plot(ci,ti)
while np.sum(c1 == c2) != 3 and len(alg) < udp.get_nix():
    rot = np.random.randint(0,6)
    out = cubes.apply_single_update_step(wi[0],rot)
    c1 = cubes.cube_position[wi[0]]
    print(c1,c2)
    alg.append(wi[0])
    alg.append(rot)
    
    #plot(cubes.cube_position,ti)

print(len(alg))
print(alg)

### Axis alignment

- works pretty well
- TODO: better halt condition

In [ ]:
wi,wt = get_wrong_cube_ids(ci,ct)
# Pick 1 coord of wrong cube and 1 coord of empty target
c1 = ci[wi[0]]
c2 = ct[wt[0]]



def find_path_to_position(id:int,c,cubes : ProgrammableCubes, budget:int = 500, verbose:bool = False):
    """
    finds path for cube with index "id" to the coordinate "c" with valid moves on "cubes"

    returns chromosomes as alg, coordinates in path, success
    """
    # Initialize variables
    alg = []
    path = []
    c1 = cubes.cube_position[id]
    c2 = c
    # Speedups
    if not is_accessible(c2,cubes.cube_position):
        return [],[],False
    
    if verbose:
        print(f"Moving cube on position {c1} to {c2}")
    # Main loop
    while np.sum(c1 == c2) != 3 and len(alg) < budget:
        # rots clockwise,counter-clockwise around x,z,y
        # 0,1 - to change y,z
        # 2,3 to change x,y
        # 4,5 to change x,z
        # idea 1
        # if x is not equal
        # rotate 2,3,4,5 and find where closer
        #dir = 0 # x
        changed_smth = False
        for dir in np.arange(0,3):
            diff = np.abs(c1 - c2)
            dist = dist_metric(c1,c2)
            if diff[dir] != 0: # change only coordinates which dont match
                for rot in ROTS_XYZ[dir]:
                    # Apply rotation
                    res = cubes.apply_single_update_step(id,rot)
                    # Compute new difference
                    c1 = cubes.cube_position[id]
                    new_dist = dist_metric(c1,c2)
                    if res == 1 and dist > new_dist: # legal and better
                        path.append(c1.copy())
                        alg.append(id)
                        alg.append(rot)
                        dist = new_dist
                        changed_smth = True
                        continue
                    elif res == 1: # legal but not better, revert
                        cubes.apply_single_update_step(id,inv_rot(rot))
                        c1 = cubes.cube_position[id]
                        continue
        if changed_smth == False: # break if all moves are tried
            break
    success = dist_metric(c1,c2) == 0
    return alg,path,success
        
id = wi[0]
alg = []
path = []
cubes.reset(ci)
for i in np.arange(len(wi)):
    tmp_alg, tmp_path, success = find_path_to_position(wi[i],ct[wt[i]],cubes)
    alg.extend(tmp_alg)
    path.extend(tmp_path)
#find_path_to_position(wi[1],wt[1],cubes,alg,path)
print(alg)
path = np.array(path)


In [ ]:
print(path)
debug_plot(np.concat([ci,path]), np.concat([ti,-np.ones(shape=path.shape[0])]))

### A*

In [ ]:
for i in cubes.cube_neighbours[0]:
    print(i)

In [ ]:
from heapq import heappush, heappop

def reconstruct_path(parents,end):
    path = [end]
    try:
        p = parents[*end][0]
    except:
        return np.array(path)
    while p:
        path.append(np.array(p))
        try:
            p = parents[*p][0]
        except:
            break
    path.reverse()
    return np.array(path)

def reconstruct_rotations(parents,end):
    chrom = []
    try:
        chrom.append(parents[*end][1])
        p = parents[*end][0]
    except:
        return np.array(chrom)
    while p:
        try:
            chrom.append(parents[*p][1])
            p = parents[*p][0]
        except:
            break
    chrom.reverse()
    return np.array(chrom)
@njit(cache=True)
def manhattan(a:np.ndarray,b:np.ndarray):
    return np.sum(np.abs(a-b))
@njit(cache=True)
def dijkstra(a:np.ndarray,b:np.ndarray):
    return 0

def astar_cubes(cubes : ProgrammableCubes, id : int, end : np.ndarray, heur : callable=manhattan, max_iter : int=np.inf):
    """ 
    cubes - working cube class
    id = id of the active cube
    end - end, target position
    heur - distance for the heuristic
    max_iter - method stops when g value higher than this is reached (means that other positions to search with
    the same value are not searched)

    f ... the heuristic + #rots to get there
    g ... #rots to get there
    parents ... stores [parent_position,rotation] - to be extracted as set of rotations afterwards
    """
    start = np.array(cubes.cube_position[id]) # ! make copy
    # Initialize arrays
    positions_to_search = []
    f = {}
    g = {}
    parents = {}

    # Initialize start
    heappush(positions_to_search,(heur(start,end),tuple(start)))
    f[tuple(start)] = 0
    g[tuple(start)] = heur(start,end)

    while len(positions_to_search) > 0:
        value, current_pos = heappop(positions_to_search)
        
        # goal check
        if np.sum(np.abs(current_pos-end)) == 0:
            cubes.apply_update_at_position(id,start) # return to original state
            return reconstruct_path(parents,end),reconstruct_rotations(parents,end),True
        

        # Escape when too many iterations
        if max_iter < g[current_pos]:
            continue

        # Move cube to currently searched position
        cubes.apply_update_at_position(id,current_pos)
        # Try all rotations, check distances, add to positions_to_search
        for rot in ROTS:
            if not cubes.apply_single_update_step(id,rot):
                continue
            new_pos = cubes.cube_position[id]

            # Get the number of steps to get to c and the previous values if exists
            tmp_g = g[current_pos] + 1
            previous_g = np.inf
            try:
                previous_g = g[tuple(new_pos)]
            except:
                pass
            #
            if previous_g == np.inf or tmp_g < previous_g:
                parents[tuple(new_pos)] = [current_pos,rot]
                g[tuple(new_pos)] = tmp_g
                f[tuple(new_pos)] = heur(new_pos,end)+tmp_g
                heappush(positions_to_search,(f,tuple(new_pos)))
            cubes.apply_single_update_step(id,inv_rot(rot),verbose=True)            
    cubes.apply_update_at_position(id,start) # return to original state
    return [],[],False

def rotations_to_chromosome(id,rotations):
    chrom = []
    for i in np.arange(len(rotations)):
        chrom.append(id)
        chrom.append(rotations[i])
    return chrom

wi,wt = get_wrong_cube_ids(ci,ct)
id = 0
cubes.reset(ci)
#print(wi[i],ct[wt[i]])
path, rotations, success = astar_cubes(cubes,wi[id],ct[wt[id]],manhattan)
print(path,rotations)
#print(len(path),len(rotations))
debug_plot(np.concat([ci,path]), np.concat([ti,-np.ones(shape=path.shape[0])]))
chrom = rotations_to_chromosome(wi[id],rotations)
cubes.reset(ci)
#print(chrom)
cubes.apply_chromosome(np.concat([chrom,[-1]]),verbose=True)
plot(cubes.cube_position,ti)


## Pairing methods

In [ ]:



def pair_colors(wi,wt,ci,ti,ct,tt):
    
    pairs = np.zeros(shape=(len(wi),4),dtype=np.int64)
    if not wi.size > 0:
        return pairs
    shuffled_wt = wt.copy()
    random.shuffle(shuffled_wt)
    pairs[:,0] = wi
    pairs[:,1] = shuffled_wt
    pairs[:,2] = ti[wi]
    pairs[:,3] = tt[shuffled_wt]
    for i in np.arange(len(wi)):
        if pairs[i,2] != pairs[i,3]:
            for j in np.arange(i+1,len(wi)):
                if pairs[i,2] == pairs[j,3]:
                    tmp_id = pairs[i,1]
                    tmp_type = pairs[i,3]
                    pairs[i,1] = pairs[j,1]
                    pairs[i,3] = pairs[j,3]
                    pairs[j,1] = tmp_id
                    pairs[j,3] = tmp_type
                    break
    return pairs


# pairs = pair_colors(wi,wt,ci,ti,ct,tt)
# print(pairs)

## REPEAT, Repeat, repeat...


### Method 1, same pairing, no colours

In [ ]:
alg = []
path = []
iter = 0
cubes.reset(ci)
ESCAPE = 0
placed_stat = []
successes = 0
while len(alg) < udp.get_nix() and iter < ESCAPE:
    # Get places which are wrong
    wi, wt = get_wrong_cube_ids(cubes.cube_position,ct)
    random.shuffle(wt)
    assert(len(wi) == len(wt)) # otherwise unsolvable
    # Pair them
    src_dest_all = [[wi[i],wt[i]] for i in np.arange(len(wi))]

    # Find paths between them
    for src_dest in src_dest_all:
        tmp_alg,tmp_path,success = find_path_to_position(src_dest[0],ct[src_dest[1]],cubes,100)
        alg.extend(tmp_alg)
        path.extend(tmp_path)
        placed_stat.append((int)(success))
        successes += (int)(success)

    iter += 1
    print(f"Budget status: {len(alg)/udp.get_nix()*100} % used, successes: {successes}")
path = np.array(path)
debug_plot(np.concat([ci,path]), np.concat([ti,-np.ones(shape=path.shape[0])]))
plt.plot(np.array(placed_stat))


### Method 2, same pairing, colours

In [ ]:
alg = []
path = []
iter = 0
cubes.reset(ci)
ESCAPE = 0
placed_stat = []
successes = 0
while len(alg) < udp.get_nix() and iter < ESCAPE:
    # Get cubes in position with wrong type
    wrong_type_ids = have_wrong_type(cubes.cube_position,ct,ti,tt)
    for id in wrong_type_ids:
        move = force_random_move(id,cubes)
        if move != -1:
            alg.extend([id,move])
    

    # Get places which are wrong
    wi, wt = get_wrong_cube_ids(cubes.cube_position,ct)
    assert(len(wi) == len(wt)) # otherwise unsolvable
    # Pair them
    src_dest_all = pair_colors(wi,wt,cubes.cube_position,ti,ct,tt)

    

    # Find paths between them
    for src_dest in src_dest_all:
        if src_dest[2] != src_dest[3]: # the place has to be correct, otherwise counterproductive
            continue
        last_positions = cubes.cube_position
        tmp_alg,tmp_path,success = find_path_to_position(src_dest[0],ct[src_dest[1]],cubes,100)
        #if success:
        alg.extend(tmp_alg)
        path.extend(tmp_path)
        # else:
        #     cubes.reset(last_positions)
        #     pass
        successes += (int)(success)
        placed_stat.append((int)(success))


    iter += 1
    print(f"mistakes:{len(wi)}+{len(wrong_type_ids)} budget status: {len(alg)/udp.get_nix()*100} % used, escape:{iter/ESCAPE*100}% successes: {successes}")
path = np.array(path)
debug_plot(np.concat([cubes.cube_position,path]), np.concat([ti,-np.ones(shape=path.shape[0])]))
plot(cubes.cube_position, ti)
plt.plot(np.array(placed_stat))


### Method 3 - good heuristic with astar, less randomization

In [ ]:
alg = []
path = []
iter = 0
cubes.reset(ci)
ESCAPE = 10
placed_stat = []
successes = 0
while len(alg) < udp.get_nix() and iter < ESCAPE:
    # Get cubes in position with wrong type
    wrong_type_ids = have_wrong_type(cubes.cube_position,ct,ti,tt)
    for id in wrong_type_ids:
        move = force_random_move(id,cubes)
        if move != -1:
            alg.extend([id,move])

    ## TODO: move wrongly placed ones first to good position, they get built around and are stuck
    

    # Get places which are wrong
    wi, wt = get_wrong_cube_ids(cubes.cube_position,ct)
    assert(len(wi) == len(wt)) # otherwise unsolvable
    # Pair them
    src_dest_all = pair_colors(wi,wt,cubes.cube_position,ti,ct,tt)

    # Find paths between them
    for src_dest in src_dest_all:
        if src_dest[2] != src_dest[3]: # the place has to be correct, otherwise counterproductive
            continue
        #last_positions = cubes.cube_position
        #tmp_alg,tmp_path,success = find_path_to_position(src_dest[0],ct[src_dest[1]],cubes,100)
        id = src_dest[0]
        start = cubes.cube_position[id]
        end = ct[src_dest[1]]
        if not is_accessible(end,cubes.cube_position):
            continue
        tmp_path,rotations,success = astar_cubes(cubes,id,end,dijkstra,50)
        #cubes.cube_position[id] = start
        #cubes.reset(cubes.cube_position)
        #cubes.reset_at_id(id,start)
        if success:
            chrom = rotations_to_chromosome(id,rotations)
            alg.extend(chrom)
            chrom = np.concatenate([chrom,[-1]])
            cubes.apply_chromosome(chrom,False)
            path.extend(tmp_path)
            #debug_plot(np.concatenate([cubes.cube_position,np.array(tmp_path)]), np.concatenate([ti,-np.ones(shape=tmp_path.shape[0])]))
            #plot(cubes.cube_position, ti)

        #     pass
        successes += (int)(success)
    placed_stat.append(successes)

    # Experiment: let all wrong cubes move 1 time randomly
    # results from few runs: overall improves fitness, maybe increases num of steps at start but helps in the end
    # wi, wt = get_wrong_cube_ids(cubes.cube_position,ct)
    # for id in wi:
    #     move = force_random_move(id,cubes)
    #     if move != -1:
    #         alg.extend([id,move])

    iter += 1
    mistakes_cnt = len(wi)+len(wrong_type_ids)
    if mistakes_cnt == 0:
        break
    print(f"mistakes:{len(wi)}+{len(wrong_type_ids)} budget status: {len(alg)/udp.get_nix()*100} % used, escape:{iter/ESCAPE*100}% successes: {successes}")
path = np.array(path)
debug_plot(np.concatenate([cubes.cube_position,path]), np.concatenate([ti,-np.ones(shape=path.shape[0])]))
#plot(cubes.cube_position, ti)
#plt.plot(np.array(placed_stat))

### Method 4: First axis, then A*

In [ ]:
alg = []
path = []
iter = 0
cubes.reset(ci)
ESCAPE = 0
placed_stat = []
successes = 0
while len(alg) < udp.get_nix() and iter < ESCAPE:
    # Get cubes in position with wrong type
    wrong_type_ids = have_wrong_type(cubes.cube_position,ct,ti,tt)
    for id in wrong_type_ids:
        move = force_random_move(id,cubes)
        if move != -1:
            alg.extend([id,move])
    

    # Get places which are wrong
    wi, wt = get_wrong_cube_ids(cubes.cube_position,ct)
    assert(len(wi) == len(wt)) # otherwise unsolvable
    # Pair them
    src_dest_all = pair_colors(wi,wt,cubes.cube_position,ti,ct,tt)

    

    # Find paths between them
    for src_dest in src_dest_all:
        if src_dest[2] != src_dest[3]: # the place has to be correct, otherwise counterproductive
            continue
        last_positions = cubes.cube_position
        tmp_alg,tmp_path,success = find_path_to_position(src_dest[0],ct[src_dest[1]],cubes,100)
        #if success:
        alg.extend(tmp_alg)
        path.extend(tmp_path)
        # else:
        #     cubes.reset(last_positions)
        #     pass
        successes += (int)(success)
        placed_stat.append((int)(success))


    iter += 1
    print(f"mistakes:{len(wi)}+{len(wrong_type_ids)} budget status: {len(alg)/udp.get_nix()*100} % used, escape:{iter/ESCAPE*100}% successes: {successes}")
iter = 0
while len(alg) < udp.get_nix() and iter < ESCAPE:
    # Get cubes in position with wrong type
    wrong_type_ids = have_wrong_type(cubes.cube_position,ct,ti,tt)
    for id in wrong_type_ids:
        move = force_random_move(id,cubes)
        if move != -1:
            alg.extend([id,move])

    ## TODO: move wrongly placed ones first to good position, they get built around and are stuck
    

    # Get places which are wrong
    wi, wt = get_wrong_cube_ids(cubes.cube_position,ct)
    assert(len(wi) == len(wt)) # otherwise unsolvable
    # Pair them
    src_dest_all = pair_colors(wi,wt,cubes.cube_position,ti,ct,tt)

    # Find paths between them
    for src_dest in src_dest_all:
        if src_dest[2] != src_dest[3]: # the place has to be correct, otherwise counterproductive
            continue
        last_positions = cubes.cube_position
        #tmp_alg,tmp_path,success = find_path_to_position(src_dest[0],ct[src_dest[1]],cubes,100)
        id = src_dest[0]
        start = cubes.cube_position[id]
        end = ct[src_dest[1]]
        if not is_accessible(end,cubes.cube_position):
            continue
        tmp_path,rotations,success = astar_cubes(cubes,id,end,manhattan,50+10*iter)
        cubes.reset(last_positions)
        if success:
            chrom = rotations_to_chromosome(id,rotations)
            alg.extend(chrom)
            chrom = np.concatenate([chrom,[-1]])
            cubes.apply_chromosome(chrom,True)
            path.extend(tmp_path)
            #debug_plot(np.concatenate([cubes.cube_position,np.array(tmp_path)]), np.concatenate([ti,-np.ones(shape=tmp_path.shape[0])]))

        #     pass
        successes += (int)(success)
    placed_stat.append(successes)

    # Experiment: let all wrong cubes move 1 time randomly
    # results from few runs: overall improves fitness, maybe increases num of steps at start but helps in the end
    # wi, wt = get_wrong_cube_ids(cubes.cube_position,ct)
    # for id in wi:
    #     move = force_random_move(id,cubes)
    #     if move != -1:
    #         alg.extend([id,move])

    iter += 1
    mistakes_cnt = len(wi)+len(wrong_type_ids)
    if mistakes_cnt == 0:
        break
    print(f"mistakes:{len(wi)}+{len(wrong_type_ids)} budget status: {len(alg)/udp.get_nix()*100} % used, escape:{iter/ESCAPE*100}% successes: {successes}")
path = np.array(path)
debug_plot(np.concatenate([cubes.cube_position,path]), np.concatenate([ti,-np.ones(shape=path.shape[0])]))
plot(cubes.cube_position, ti)
plt.plot(np.array(placed_stat))

## Evaluation of the chromosome

In [ ]:

chrom = alg.copy()
#chrom = [1,0]
chrom = np.array(chrom)
def remove_illegal_moves_from_chromosome(chromosome):
    """ From tutorial 1"""
    cubes = ProgrammableCubes(ci)

    filtered_chromosome = []

    for i in range(int(len(chromosome)/2)):
        cube_id = chromosome[i*2]
        move = chromosome[i*2+1]
        done = cubes.apply_single_update_step(cube_id, move)
        # done is 1 if the move is legal and 0 otherwise
        if done == 1:
            filtered_chromosome += [cube_id, move]
    # The part to be evaluated by the fitness function ends here,
    # thus we add -1
    filtered_chromosome += [-1]
    
    # Fill up the remaining chromosome with 0s (will not be evaluated)
    if len(filtered_chromosome) < udp.get_nix():
        for i in range(udp.get_nix() - 1 - len(filtered_chromosome)):
            filtered_chromosome += [0]
        # Mandatory entry of -1 at the very end
        filtered_chromosome += [-1]
        
    return np.array(filtered_chromosome)
print(len(chrom))
print(chrom)
chrom = remove_illegal_moves_from_chromosome(chrom)
print(f" Achieved fitness: {udp.fitness(chrom)}")
print("Initial config")
plot(ci,ti)
print("Achieved config")
udp.plot("ensemble",types)
print("Target config")
udp.plot("target",types)





## Export chromosome

In [ ]:
#
np.save("chromosome",chrom)